# 18 — Preliminary check: does forward simulation beat simple base-stock?

## What this notebook does

Notebook 17 asked whether **more store data** improves filter accuracy and profit
when ordering uses **base-stock only** (no simulated futures). This notebook asks
a separate controller question:

> Holding the **oracle-shelf controller world** fixed (SIM-01=B), does **looking
> ahead with Monte Carlo rollout** beat **simple base-stock** on the same random
> seeds and simulated week?

**Important:** `rollout_eval` uses the **oracle-shelf** controller world — the
policy sees perfect on-shelf state (B-state), **not** the filtered beliefs from
notebook 17. We are isolating the value of forward simulation, not re-testing data
packages.

Rollout is expensive in the cloud (many forward simulations per order day). This
notebook runs a **paired, medium** comparison: four seeds, 14 scored days, horizon
7 with 4 sample paths.

## Context: tuned policy parameters

An earlier Bayesian-optimization run (saved as `outputs/sw_alpha_bo.json` when
present) searched for good values of:

- **Order smoothing** — how aggressively orders react to belief updates
- **Waste penalty weight** — trade-off between profit, waste, and stockouts

If that file exists, we reuse its validation seeds and best profit smoothing (α).
This notebook **does not re-run optimization** and **does not sweep an alpha grid**.
It compares two policies at a single α:

| Policy | What it does in this run |
| --- | --- |
| **Base-stock only** | Standard damped base-stock; no forward simulation |
| **With rollout** | Same oracle beliefs, but evaluates a 7-day horizon with 4 sample paths |

Set `SMOKE=True` to shrink to one seed, one arm, and three scored days for plumbing.

## How to run

Same setup as notebook 17 (repo root, built wheel, Modal login). Set
`BATCH_MODE = "local"` to run on a laptop without Modal.

In [ ]:
from __future__ import annotations

import json
import matplotlib.pyplot as plt
import os
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for _candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (_candidate / "src" / "blueberries_voi").is_dir():
        REPO_ROOT = _candidate
        break

_wheel_dir = REPO_ROOT / "dist" / "wheel"
_gsin_bin = REPO_ROOT / "target" / "release" / "examples" / "gsin_upc_diag"
os.environ.setdefault("BLUEBERRIES_VOI_BACKEND", "rust")
if _wheel_dir.is_dir():
    os.environ["BLUEBERRIES_VOI_WHEEL"] = str(_wheel_dir)
if _gsin_bin.is_file():
    os.environ["GSIN_UPC_DIAG_BIN"] = str(_gsin_bin)

from blueberries_voi.experiments.modal_dispatch import run_batch

BATCH_MODE: Literal["modal", "local"] = "modal"
SMOKE = False

POLICY_LABELS = {
    "sw": "Base-stock only",
    "rollout": "With rollout",
}

BO_JSON = REPO_ROOT / "outputs" / "sw_alpha_bo.json"
bo_meta: dict = {}
if BO_JSON.is_file():
    bo_meta = json.loads(BO_JSON.read_text(encoding="utf-8"))
    print(f"Loaded {BO_JSON.name}: arm={bo_meta.get('arm')} full_run={bo_meta.get('full_run')}")
else:
    print(f"No {BO_JSON.name}; using fixed defaults below")

VAL_SEEDS = tuple(bo_meta.get("val_seeds", (42, 7, 101, 2024))[:4]) if bo_meta else (42, 7, 101, 2024)
ALPHA = float(bo_meta.get("best_alpha_profit_soo", 0.9)) if bo_meta else 0.9
N_BURN, N_SCORE = 2, 14
ROLLOUT_H, N_PATHS, RADIUS = 7, 4, 1
RHO = float(bo_meta.get("best_rho_profit_soo", 0.8))

## Paired policy comparison

Each row is one **(random seed, policy)** episode with identical simulated weather,
demand, and spoilage. Settings for this run:

| Parameter | Value | Note |
| --- | --- | --- |
| Controller world | Oracle shelf (SIM-01=B) | Perfect on-shelf beliefs; **not** nb17 filter packages |
| Random seeds | Four from tuning file or `(42, 7, 101, 2024)` | Same stream per seed across policies |
| Order smoothing | From `best_alpha_profit_soo` or 0.9 | Fixed; no alpha grid here |
| Waste penalty weight | From tuning file or 0.8 | Profit vs waste trade-off |
| Warm-up / scored | 2 / 14 days | Longer than smoke, shorter than notebook 16 bakeoff |
| Rollout horizon | 7 days, 4 sample paths | Prelim budget; production bakeoff uses H=28, paths=8 |

The table below averages profit, waste, and stockouts by policy. The next section
subtracts base-stock profit from rollout profit **per seed** (paired difference).

In [ ]:
rows = run_batch(
    "rollout_eval",
    BATCH_MODE,
    smoke=SMOKE,
    seeds=VAL_SEEDS,
    arms=("sw", "rollout"),
    alphas=(ALPHA,),
    rho=RHO,
    n_burn=N_BURN,
    n_score=N_SCORE,
    rollout_h=ROLLOUT_H,
    n_rollout_paths=N_PATHS,
    candidate_case_radius=RADIUS,
)
df = pd.DataFrame(rows)
df["policy"] = df["arm_id"].map(POLICY_LABELS)
summary = df.groupby("policy")[["profit", "waste", "stockout"]].mean()
summary

In [ ]:
pivot = df.pivot(index="seed", columns="arm_id", values="profit").sort_index()
pivot = pivot.rename(columns=POLICY_LABELS)
summary_named = summary.rename(index=POLICY_LABELS)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

pivot.plot.bar(ax=axes[0], color={"Base-stock only": "steelblue", "With rollout": "darkorange"})
axes[0].set_title(f"Scored profit by seed ({N_SCORE} days, paired)")
axes[0].set_ylabel("profit")
axes[0].legend(title="policy")
axes[0].tick_params(axis="x", rotation=0)

summary_named.plot.bar(ax=axes[1], y="profit", legend=False, color=["steelblue", "darkorange"])
axes[1].set_title(f"Mean profit across {len(VAL_SEEDS)} seeds")
axes[1].set_ylabel("profit")

fig.suptitle(
    f"Rollout vs base-stock — oracle shelf (α={ALPHA}, ρ={RHO:.2f}, H={ROLLOUT_H}, paths={N_PATHS})",
    y=1.05,
)
fig.tight_layout()
plt.show()

### What to look for

**Modal run (2026-08-24):** 6 rollout shards (3 BO validation seeds × 2 arms), ~59 s wall time. Settings: oracle shelf, α=0.531, ρ=0.698 (from `outputs/sw_alpha_bo.json`), H=7, 4 paths, 14 scored days.

| Policy | Mean profit | Mean waste | Mean stockout |
| --- | ---: | ---: | ---: |
| Base-stock only | 188.2 | 17.7 | 130.0 |
| With rollout | **306.7** | 22.0 | 105.0 |

Rollout raised mean profit by **~$118** on every validation seed while trading slightly more waste for fewer stockouts. At this budget the policies are **not** tied — rollout clearly wins on these three seeds.

**Caveats:** only **three** BO validation seeds (not four), horizon 7 / 4 paths (much smaller than notebook 16's H=28 / 8 paths), and oracle-shelf beliefs. Treat as a pipeline check, not a production rollout claim.


## Paired profit difference (rollout minus base-stock)

For each random seed we subtract base-stock profit from rollout profit on the
**same simulated week**. A positive value means rollout helped; zero means no
detectable change at this budget.

In [ ]:
sw = df[df["arm_id"] == "sw"].set_index("seed")["profit"]
ro = df[df["arm_id"] == "rollout"].set_index("seed")["profit"]
delta = (ro - sw).dropna()
mean_delta = float(delta.mean())
sem = float(delta.std(ddof=1) / np.sqrt(len(delta))) if len(delta) > 1 else float("nan")
print(
    f"paired Δprofit (rollout - sw): mean={mean_delta:.2f} "
    f"SEM={sem:.2f} n={len(delta)} α={ALPHA} rho={RHO}"
)

### Paired difference — what the printout means

**This run:** paired Δprofit (rollout − base-stock) = **+118.5** (SEM **11.7**, **n=3** seeds from `sw_alpha_bo.json`).

Per-seed deltas: **+100.5**, **+114.5**, **+140.5** — rollout helped on every seed, not a single lucky draw.

Interpretation guide:

- **Mean Δ ≫ 0** — rollout added substantial profit at this budget on oracle-shelf beliefs.
- **SEM ~12 on n=3** — direction is clear; magnitude still has wide uncertainty.
- **Three BO validation seeds** — better than a smoke test, still far from notebook 16's bakeoff scale.

A large positive mean here is a valid **plumbing** outcome for the Modal path; it does **not** by itself justify production rollout settings without the longer horizon and seed grid in notebook 16.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
delta.sort_index().plot.bar(ax=ax, color="slategray")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_title("Paired profit difference by seed\n(rollout minus base-stock)")
ax.set_ylabel("profit difference")
fig.tight_layout()
plt.show()

### Conclusion

This notebook tests whether **forward simulation** adds value over **base-stock** in the **oracle-shelf** controller world (SIM-01=B), independent of notebook 17's data package ladder.

**2026-08-24 Modal result:** at three BO validation seeds, 14 scored days, and H=7 / paths=4, rollout beat base-stock by **+$118.5** mean paired profit (all three seeds positive). That is a clear separation at this prelim budget — unlike a near-zero tie.

For a production bakeoff, use notebook 16's path: alpha grid, H=28, paths=8, and many more seeds. See `experiments/modal/PRELIM_SCALING.md` for staged budget increases.
